In [1]:
!pip install biopython transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.1 MB/s eta 0:00:00


In [2]:
%%bash
# Install local HMMER suite and PyTorch/Transformers
sudo apt-get update -y -qq
sudo apt-get install -y hmmer -qq
pip install -q transformers torch

# Install DIAMOND aligner
wget -q https://github.com/bbuchfink/diamond/releases/download/v2.1.8/diamond-linux64.tar.gz
tar xzf diamond-linux64.tar.gz

# Download and format the Swiss-Prot database for DIAMOND
wget -q ftp://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz
gunzip uniprot_sprot.fasta.gz
./diamond makedb --in uniprot_sprot.fasta -d swissprot --quiet

# Download and format the Pfam database for HMMER
wget -q ftp://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.hmm.gz
gunzip Pfam-A.hmm.gz
hmmpress Pfam-A.hmm

Selecting previously unselected package libdivsufsort3:amd64.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../libdivsufsort3_2.0.1-6_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-6) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.4+dfsg-2_amd64.deb ...
Unpacking hmmer (3.4+dfsg-2) ...
Setting up libdivsufsort3:amd64 (2.0.1-6) ...
Setting up hmmer (3.4+dfsg-2) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.rea

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [4]:
import asyncio
import os
import tempfile

import torch
from transformers import AutoTokenizer, EsmModel
from Bio.SeqUtils.ProtParam import ProteinAnalysis


class EndToEndBioinformaticsPipeline:
    def __init__(self):
        self.DIAMOND_IDENTITY_THRESHOLD = 30.0
        self.DIAMOND_QUERY_COVERAGE_THRESHOLD = 80.0
        self.HMMER_EVAL_THRESHOLD = 1e-10
        model_name = "facebook/esm2_t6_8M_UR50D"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = EsmModel.from_pretrained(model_name)

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )
        self.model.to(self.device)
        self.model.eval()

    async def _run_diamond(self, fasta_path: str) -> dict:
        out_path = f"{fasta_path}.tsv"

        try:
            cmd = [
                "./diamond",
                "blastp",
                "-d", "swissprot.dmnd",
                "-q", fasta_path,
                "-o", out_path,
                "--outfmt", "6",
                "qseqid", "sseqid", "pident", "evalue",
                "bitscore", "qstart", "qend", "sstart", "send",
                "qlen", "slen",
                "--max-target-seqs", "1",
                "--quiet",
            ]

            process = await asyncio.create_subprocess_exec(
                *cmd,
                stdout=asyncio.subprocess.DEVNULL,
                stderr=asyncio.subprocess.PIPE,
            )
            _, stderr = await process.communicate()

            if process.returncode != 0:
                error = stderr.decode(errors="replace").strip()
                return {
                    "source": "DIAMOND",
                    "identity": 0.0,
                    "best_hit": "None",
                    "e_value": 1.0,
                    "query_coverage": 0.0,
                    "error": error,
                }

            if not os.path.exists(out_path) or os.path.getsize(out_path) == 0:
                return {
                    "source": "DIAMOND",
                    "identity": 0.0,
                    "best_hit": "None",
                    "e_value": 1.0,
                    "query_coverage": 0.0,
                }

            with open(out_path, "r") as f:
                parts = f.readline().strip().split("\t")

            if len(parts) < 11:
                return {
                    "source": "DIAMOND",
                    "identity": 0.0,
                    "best_hit": "None",
                    "e_value": 1.0,
                    "query_coverage": 0.0,
                    "error": "Unexpected DIAMOND output format",
                }

            identity = float(parts[2])
            e_value = float(parts[3])

            qstart = int(parts[6])
            qend = int(parts[7])
            qlen = int(parts[10])

            query_coverage = (
                100.0 * (abs(qend - qstart) + 1) / qlen
                if qlen > 0 else 0.0
            )

            return {
                "source": "DIAMOND",
                "best_hit": parts[1],
                "identity": identity,
                "e_value": e_value,
                "query_coverage": query_coverage,
            }

        except FileNotFoundError as e:
            return {
                "source": "DIAMOND",
                "identity": 0.0,
                "best_hit": "None",
                "e_value": 1.0,
                "query_coverage": 0.0,
                "error": str(e),
            }

        finally:
            if os.path.exists(out_path):
                os.remove(out_path)

    async def _run_hmmer(self, fasta_path: str) -> dict:
        out_path = f"{fasta_path}.domtblout"

        try:
            cmd = [
                "hmmscan",
                "--noali",
                "--domtblout", out_path,
                "Pfam-A.hmm",
                fasta_path,
            ]

            process = await asyncio.create_subprocess_exec(
                *cmd,
                stdout=asyncio.subprocess.DEVNULL,
                stderr=asyncio.subprocess.PIPE,
            )
            _, stderr = await process.communicate()

            if process.returncode != 0:
                return {
                    "source": "HMMER",
                    "domain": "None",
                    "accession": "None",
                    "e_value": 1.0,
                    "error": stderr.decode(errors="replace").strip(),
                }

            best_hit = None

            if os.path.exists(out_path):
                with open(out_path, "r") as f:
                    for line in f:
                        if line.startswith("#"):
                            continue

                        parts = line.split()
                        if len(parts) < 22:
                            continue

                        hit = {
                            "source": "HMMER",
                            "domain": parts[0],
                            "accession": parts[1],
                            "e_value": float(parts[12]),
                        }

                        if best_hit is None or hit["e_value"] < best_hit["e_value"]:
                            best_hit = hit

            if best_hit is not None:
                return best_hit

            return {
                "source": "HMMER",
                "domain": "None",
                "accession": "None",
                "e_value": 1.0,
            }

        except FileNotFoundError as e:
            return {
                "source": "HMMER",
                "domain": "None",
                "accession": "None",
                "e_value": 1.0,
                "error": str(e),
            }

        finally:
            if os.path.exists(out_path):
                os.remove(out_path)

    def _extract_esm2_embedding(self, sequence: str) -> dict:
        inputs = self.tokenizer(
            sequence,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
        token_embeddings = outputs.last_hidden_state[:, 1:-1, :]

        pooled = token_embeddings.mean(dim=1).squeeze(0)
        embedding = pooled.cpu().numpy()

        return {
            "source": "ESM-2",
            "embedding_dimensions": int(embedding.shape[0]),
            "embedding": embedding,
        }

    def _run_protparam_analysis(self, sequence: str) -> dict:
        analyzer = ProteinAnalysis(sequence)
        sec_structure = analyzer.secondary_structure_fraction()

        return {
            "Molecular Weight (kDa)": round(
                analyzer.molecular_weight() / 1000, 2
            ),
            "Isoelectric Point (pI)": round(
                analyzer.isoelectric_point(), 2
            ),
            "Instability Index (heuristic)": round(
                analyzer.instability_index(), 2
            ),
            "GRAVY (mean hydropathy)": round(
                analyzer.gravy(), 3
            ),
            "Alpha Helix Propensity Fraction": round(
                sec_structure[0], 3
            ),
            "Beta Sheet Propensity Fraction": round(
                sec_structure[2], 3
            ),
        }

    def _calculate_consensus(
        self,
        diamond_res: dict,
        hmmer_res: dict,
        esm_res: dict,
    ) -> dict:
        score = 0
        summary = []

        identity = diamond_res.get("identity", 0.0)
        coverage = diamond_res.get("query_coverage", 0.0)

        if (
            identity > self.DIAMOND_IDENTITY_THRESHOLD
            and coverage >= self.DIAMOND_QUERY_COVERAGE_THRESHOLD
        ):
            score += 50
            summary.append(
                f"[+] DIAMOND: {identity:.2f}% identity and "
                f"{coverage:.2f}% query coverage to "
                f"{diamond_res.get('best_hit', 'Unknown')}."
            )
        else:
            summary.append(
                f"[-] DIAMOND: identity={identity:.2f}%, "
                f"query coverage={coverage:.2f}%; "
                "the configured thresholds were not both met."
            )

        hmm_evalue = hmmer_res.get("e_value", 1.0)

        if hmm_evalue < self.HMMER_EVAL_THRESHOLD:
            score += 40
            summary.append(
                f"[+] HMMER: detected {hmmer_res.get('domain')} "
                f"(independent domain E-value: {hmm_evalue:.2e})."
            )
        else:
            summary.append(
                "[-] HMMER: no domain passed the configured E-value threshold."
            )

        if esm_res.get("embedding_dimensions") == 320:
            summary.append(
                "[+] ESM-2: successfully extracted a 320-dimensional "
                "sequence embedding."
            )

        return {
            "evidence_score": score,
            "maximum_score": 90,
            "summary": summary,
        }

    def print_report(self, consensus: dict, physio: dict):
        print("\nSUMMARY")
        for line in consensus["summary"]:
            print(line)

        print("\nSEQUENCE-DERIVED PHYSICOCHEMICAL PROPERTIES")
        for key, value in physio.items():
            print(f"{key:<38}: {value}")

        print("\nINTERPRETATION")

        instability = physio["Instability Index (heuristic)"]
        if instability > 40:
            print(
                "* Instability index is above 40. This is a sequence-based "
                "heuristic, not a direct measurement of protein stability."
            )
        else:
            print(
                "* Instability index is at or below 40. This heuristic alone "
                "does not establish experimental stability."
            )

        gravy = physio["GRAVY (mean hydropathy)"]
        if gravy > 0:
            print(
                "* Positive GRAVY indicates a hydrophobic sequence on average. "
                "It does not by itself establish membrane localization."
            )
        else:
            print(
                "* Negative GRAVY indicates a hydrophilic sequence on average. "
                "It does not by itself establish cellular localization or solubility."
            )

        pI = physio["Isoelectric Point (pI)"]
        if pI < 7.4:
            print(
                "* At pH 7.4, the protein is expected to have a net negative "
                "charge based on its theoretical pI. Anion-exchange binding "
                "is possible, but must be tested experimentally."
            )
        elif pI > 7.4:
            print(
                "* At pH 7.4, the protein is expected to have a net positive "
                "charge based on its theoretical pI. Cation-exchange binding "
                "is possible, but must be tested experimentally."
            )
        else:
            print(
                "* The theoretical pI is approximately 7.4, so the net charge "
                "may be close to zero at that pH."
            )



    async def execute(self, sequence: str):
        sequence = sequence.strip().upper()

        if not sequence:
            raise ValueError("Protein sequence cannot be empty.")

        valid_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
        invalid = set(sequence) - valid_amino_acids
        if invalid:
            raise ValueError(
                f"Invalid amino-acid characters found: {sorted(invalid)}"
            )

        with tempfile.NamedTemporaryFile(
            mode="w",
            delete=False,
            suffix=".fasta"
        ) as tmp:
            tmp.write(f">query\n{sequence}\n")
            fasta_path = tmp.name

        try:
            print(
                "Analyzing sequence: running DIAMOND, HMMER, "
                "ESM-2, and physicochemical analysis..."
            )

            diamond_task = self._run_diamond(fasta_path)
            hmmer_task = self._run_hmmer(fasta_path)
            esm_task = asyncio.to_thread(
                self._extract_esm2_embedding, sequence
            )
            physio_task = asyncio.to_thread(
                self._run_protparam_analysis, sequence
            )

            diamond_res, hmmer_res, esm_res, physio_res = await asyncio.gather(
                diamond_task,
                hmmer_task,
                esm_task,
                physio_task,
            )

            consensus = self._calculate_consensus(
                diamond_res,
                hmmer_res,
                esm_res,
            )

            self.print_report(consensus, physio_res)
            # return {
            #     "diamond": diamond_res,
            #     "hmmer": hmmer_res,
            #     "esm": {
            #         "source": esm_res["source"],
            #         "embedding_dimensions": esm_res["embedding_dimensions"],
            #         "embedding": esm_res["embedding"],
            #     },
            #     "physicochemical": physio_res,
            #     "consensus": consensus,
            # }

        finally:
            if os.path.exists(fasta_path):
                os.remove(fasta_path)

# Testing

In [5]:
test_sequence = "MSDVAIVKEGWLHKRGEYIKTWRPRYFLLKNDGTFIGYKERPQDVDQREAPLNNFSVAQCQLMKTERPRPNTFIIRCLQWTTVIERTFHVETPEEREEWTTAIQTVADGLKKQEEEEMDFRSGSPSDNSGAEEMEVSLAKPKHRVTMNEFEYLKLLGKGTFGKVILVKEKATGRYYAMKILKKEVIVAKDEVAHTLTENRVLQNSRHPFLTALKYSFQTHDRLCFVMEYANGGELFFHLSRERVFSEDRARFYGAEIVSALDYLHSEKNVVYRDLKLENLMLDKDGHIKITDFGLCKEGIKDGATMKTFCGTPEYLAPEVLEDNDYGRAVDWWGLGVVMYEMMCGRLPFYNQDHEKLFELILMEEIRFPRTLGPEAKSLLSGLLKKDPKQRLGGGSEDAKEIMQHRFFAGIVWQHVYEKKLSPPFKPQVTSETDTRYFDEEFTAQMITITPPDQDDSMECVDSERRPHFPQFSYSASGTA"
pipeline = EndToEndBioinformaticsPipeline()
await pipeline.execute(test_sequence)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 31.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Analyzing sequence: running DIAMOND, HMMER, ESM-2, and physicochemical analysis...

SUMMARY
[+] DIAMOND: 100.00% identity and 100.00% query coverage to sp|P31749|AKT1_HUMAN.
[+] HMMER: detected Pkinase (independent domain E-value: 1.90e-79).
[+] ESM-2: successfully extracted a 320-dimensional sequence embedding.

SEQUENCE-DERIVED PHYSICOCHEMICAL PROPERTIES
Molecular Weight (kDa)                : 55.69
Isoelectric Point (pI)                : 5.75
Instability Index (heuristic)         : 35.47
GRAVY (mean hydropathy)               : -0.575
Alpha Helix Propensity Fraction       : 0.348
Beta Sheet Propensity Fraction        : 0.356

INTERPRETATION
* Instability index is at or below 40. This heuristic alone does not establish experimental stability.
* Negative GRAVY indicates a hydrophilic sequence on average. It does not by itself establish cellular localization or solubility.
* At pH 7.4, the protein is expected to have a net negative charge based on its theoretical pI. Anion-exchange bind

In [6]:
test_sequence = "MLELLPTAVEGVSQAQITGRPEWIWLALGTALMGLGTLYFLVKGMGVSDPDAKKFYAITTLVPAIAFTMYLSMLLGYGLTMVPFGGEQNPIYWARYADWLFTTPLLLLDLALLVDADQGTILALVGADGIMIGTGLVGALTKVYSYRFVWWAISTAAMLYILYVLFFGFTSKAESMRPEVASTFKVLRNVTVVLWSAYPVVWLIGSEGAGIVPLNIETLLFMVLDVSAKVGFGLILLRSRAIFGEAEAPEPSAGDGAAATSD"
pipeline = EndToEndBioinformaticsPipeline()
await pipeline.execute(test_sequence)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Analyzing sequence: running DIAMOND, HMMER, ESM-2, and physicochemical analysis...

SUMMARY
[+] DIAMOND: 100.00% identity and 100.00% query coverage to sp|P02945|BACR_HALSA.
[+] HMMER: detected Bac_rhodopsin (independent domain E-value: 2.70e-74).
[+] ESM-2: successfully extracted a 320-dimensional sequence embedding.

SEQUENCE-DERIVED PHYSICOCHEMICAL PROPERTIES
Molecular Weight (kDa)                : 28.26
Isoelectric Point (pI)                : 4.62
Instability Index (heuristic)         : 27.03
GRAVY (mean hydropathy)               : 0.723
Alpha Helix Propensity Fraction       : 0.37
Beta Sheet Propensity Fraction        : 0.489

INTERPRETATION
* Instability index is at or below 40. This heuristic alone does not establish experimental stability.
* Positive GRAVY indicates a hydrophobic sequence on average. It does not by itself establish membrane localization.
* At pH 7.4, the protein is expected to have a net negative charge based on its theoretical pI. Anion-exchange binding is pos

In [7]:
test_sequence = "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHECSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELPPGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPGGSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD"
pipeline = EndToEndBioinformaticsPipeline()
await pipeline.execute(test_sequence)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Analyzing sequence: running DIAMOND, HMMER, ESM-2, and physicochemical analysis...

SUMMARY
[+] DIAMOND: 99.70% identity and 99.75% query coverage to sp|P04637|P53_HUMAN.
[+] HMMER: detected P53 (independent domain E-value: 3.00e-62).
[+] ESM-2: successfully extracted a 320-dimensional sequence embedding.

SEQUENCE-DERIVED PHYSICOCHEMICAL PROPERTIES
Molecular Weight (kDa)                : 43.5
Isoelectric Point (pI)                : 6.2
Instability Index (heuristic)         : 74.88
GRAVY (mean hydropathy)               : -0.747
Alpha Helix Propensity Fraction       : 0.301
Beta Sheet Propensity Fraction        : 0.265

INTERPRETATION
* Instability index is above 40. This is a sequence-based heuristic, not a direct measurement of protein stability.
* Negative GRAVY indicates a hydrophilic sequence on average. It does not by itself establish cellular localization or solubility.
* At pH 7.4, the protein is expected to have a net negative charge based on its theoretical pI. Anion-exchange 

In [8]:
test_sequence = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
pipeline = EndToEndBioinformaticsPipeline()
await pipeline.execute(test_sequence)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Analyzing sequence: running DIAMOND, HMMER, ESM-2, and physicochemical analysis...

SUMMARY
[+] DIAMOND: 100.00% identity and 100.00% query coverage to sp|Q865C5|UBIQ_CAMDR.
[+] HMMER: detected ubiquitin (independent domain E-value: 7.70e-34).
[+] ESM-2: successfully extracted a 320-dimensional sequence embedding.

SEQUENCE-DERIVED PHYSICOCHEMICAL PROPERTIES
Molecular Weight (kDa)                : 8.56
Isoelectric Point (pI)                : 6.56
Instability Index (heuristic)         : 36.06
GRAVY (mean hydropathy)               : -0.489
Alpha Helix Propensity Fraction       : 0.329
Beta Sheet Propensity Fraction        : 0.395

INTERPRETATION
* Instability index is at or below 40. This heuristic alone does not establish experimental stability.
* Negative GRAVY indicates a hydrophilic sequence on average. It does not by itself establish cellular localization or solubility.
* At pH 7.4, the protein is expected to have a net negative charge based on its theoretical pI. Anion-exchange bin